In [52]:
# ## linking institutions from mag with list of all institutions

import subprocess
import sqlite3 as sqlite
import argparse
import os
import pandas as pd
import numpy as np 

print(os.getcwd())


from src.dataprep.helpers.functions import *
from src.dataprep.helpers.variables import *

con=sqlite.connect(db_file)



/home/flavio/projects/mag_sample


### 1. Import all data 

In [53]:
# ## mag sample: Check institutions and names
query="""select AffiliationId, NormalizedName, PublicationCount
         from affiliations
         inner join affiliation_outcomes using(AffiliationId)
         where iso3166code = 'US' """
mag=pd.read_sql(sql=query, con=con)

# ## cng_institutions sample
query= "select * from cng_institutions"
cng_institutions=pd.read_sql(sql=query, con=con)
cng_institutions = cng_institutions.loc[: , ["unitid", "normalized_name", "iclevel", "stabbr"]] 

# ## proquest sample
query="select university_id, originalname, normalizedname, location from pq_unis where location like '%United States%' "
proquest=pd.read_sql(sql=query, con=con)


In [54]:
mag.head()

,AffiliationId,NormalizedName,PublicationCount
0,50614327,amec foster wheeler,678
1,80046288,walden university,1113
2,94975175,des moines university,764
3,158506100,pennsylvania board of probation and parole,3
4,192633058,university of saint joseph,545


In [55]:
cng_institutions.head()

,unitid,normalized_name,iclevel,stabbr
0,177834,a t still university of health sciences,1,MO
1,134811,ai miami international university of art and d...,1,FL
2,429094,aoma graduate school of integrative medicine,1,TX
3,404994,asa college,1,NY
4,446127,ata career education,2,FL


In [56]:
proquest.head()

,university_id,originalname,normalizedname,location
0,1,University of Massachusetts Amherst,university of massachusetts amherst,United States -- Massachusetts
1,2,Northwestern University,northwestern university,United States -- Illinois
2,4,"University of California, Los Angeles",university of california los angeles,United States -- California
3,7,University of Washington,university of washington,United States -- Washington
4,9,Old Dominion University,old dominion university,United States -- Virginia


### 2. Prepare data 

In [57]:
# ## institutions: drop Puerto Rico
    # NOTE: use .loc for filtering. https://stackoverflow.com/questions/23296282/what-rules-does-pandas-use-to-generate-a-view-vs-a-copy
cng_institutions= cng_institutions.loc[cng_institutions.stabbr != "PR", :] 
print(cng_institutions.count())


unitid             3853
normalized_name    3853
iclevel            3853
stabbr             3853
dtype: int64


In [58]:
# do not include for now
# adjusting the name doesn't change anything
# excl={"of", "and", "for", "the"}
# for char in excl:
#     normalized_name=institutions.normalized_name.replace(char, "")
#     NormalizedName=mag.NormalizedName.replace(char, "")

### 3. Link 

The workflow should do the following.

#### Linking

We need to find the state of the institution in MAG. This is reported, at least for some, in parenthesis. For instance, "Westminster College (Utah)". 
But other places ("Princeton University") do not have this information. 
Therefore, we need to link with 2 passes, and split the MAG affiliations by whether they have a string in parentheses or not.
- for those mag institutions with parentheses in the name: separate the name column "Westminster College (Utah)" into 2: name = "Westminster College", state = "Utah". Then define a correspondence from state names to state abbreviations (either way is ok, but make it reproducible in the script). Then merge to cng_institutions on name and state/stabbr.
- for those mag institutions without parentheses in the name, merge to cng_institutions by name (as per below). But make sure first that you drop all entities already linked in the previous step from both mag and cng_institutions, as otherwise there will be trouble later on.


Then, stack the two linked dataframes together and add a column of how the entities were linked.


##### Inspecting
- Check for duplicates: report duplicates in the notebook. we will have to deal with them in a next iteration.
- Inspect non-matches: look at the entities in mag with the highest number of publications. manually look up whether there is a record with a similar name in `institutions`. What is different? should we adjust the names to improve matching? --> make a list of the steps you think are most important/bring the highest gains in this respect. we can then discuss the ideas. 


In [59]:
# Here is some code for the suggested next steps
    # the main goal is to get rid of non-research outlets in mag. thus, change the merge direction

mag_inst = mag.merge(cng_institutions, how = "outer", indicator = "origin", left_on = "NormalizedName", right_on = "normalized_name")
# dt.head()
mag_inst.head()




,AffiliationId,NormalizedName,PublicationCount,unitid,normalized_name,iclevel,stabbr,origin
0,50614327.0,amec foster wheeler,678.0,NaN,NaN,NaN,NaN,left_only
1,80046288.0,walden university,1113.0,125231.0,walden university,1.0,MN,both
2,94975175.0,des moines university,764.0,NaN,NaN,NaN,NaN,left_only
3,158506100.0,pennsylvania board of probation and parole,3.0,NaN,NaN,NaN,NaN,left_only
4,192633058.0,university of saint joseph,545.0,130314.0,university of saint joseph,1.0,CT,both


#### Are there duplicated links?

In [60]:
linked_inst = mag_inst.loc[mag_inst.origin == "both"]
print(f"We have {linked_inst.shape[0]} entries in linked_inst")
print(f"We have {linked_inst['AffiliationId'].nunique()} unique mag affiliations in linked_inst")

# see duplicated records
linked_inst.loc[linked_inst.duplicated(subset = "AffiliationId")].sort_values(by = "PublicationCount", ascending = False).head(10)


We have 2146 entries in mag_linked
We have 2101 unique mag affiliations in mag_linked


,AffiliationId,NormalizedName,PublicationCount,unitid,normalized_name,iclevel,stabbr,origin
3242,161515732.0,university of st thomas,3145.0,227863.0,university of st thomas,1.0,TX,both
3025,895819238.0,bethel university,2975.0,173160.0,bethel university,1.0,MN,both
3026,895819238.0,bethel university,2975.0,219718.0,bethel university,1.0,TN,both
6978,177328695.0,union college,2854.0,181738.0,union college,1.0,NE,both
6979,177328695.0,union college,2854.0,196866.0,union college,1.0,NY,both
6474,33821262.0,lincoln university,1660.0,213598.0,lincoln university,1.0,PA,both
6473,33821262.0,lincoln university,1660.0,177940.0,lincoln university,1.0,MO,both
2724,73236664.0,wheaton college,1398.0,168281.0,wheaton college,1.0,MA,both
4049,193478522.0,emmanuel college,629.0,165671.0,emmanuel college,1.0,MA,both
2722,53618229.0,wheaton college,595.0,168281.0,wheaton college,1.0,MA,both


In [61]:
print(mag.shape)
print(mag_inst.shape) # do we get multiple links? why? -->  this should be a minor problem when merging also on state
mag_inst.groupby(["origin"]).size()



(9085, 3)
(10859, 8)


origin
left_only     6984
right_only    1729
both          2146
dtype: int64

In [62]:
# inspect duplicated names within state in cng:
cng_institutions.loc[cng_institutions.duplicated(subset = ["stabbr", "normalized_name"])]
# so 5 institutions have duplicates within state-name; only one of them has iclevel = 1.

,unitid,normalized_name,iclevel,stabbr
1524,440776,interactive college of technology,2,TX
1525,443696,interactive college of technology,2,TX
1706,408729,laurel technical institute,2,PA
2969,446552,southern technical college,2,FL
3798,224679,western technical college,1,TX


In [63]:
mag_inst.loc[mag_inst.origin == "left_only"].sort_values(by = "PublicationCount", ascending = False).head(10)
    # here, we should certainly find univ of michigan, univ of minnesota etc in `institutions`


,AffiliationId,NormalizedName,PublicationCount,unitid,normalized_name,iclevel,stabbr,origin
4278,1.982037e+07,chinese academy of sciences,585146.0,NaN,NaN,NaN,NaN,left_only
1972,2.783732e+07,university of michigan,302259.0,NaN,NaN,NaN,NaN,left_only
9068,1.299303e+09,national institutes of health,288274.0,NaN,NaN,NaN,NaN,left_only
4079,2.014487e+08,university of washington,275598.0,NaN,NaN,NaN,NaN,left_only
5686,1.613188e+08,university of california los angeles,256110.0,NaN,NaN,NaN,NaN,left_only
6298,1.302385e+08,university of minnesota,231829.0,NaN,NaN,NaN,NaN,left_only
3508,1.288882e+09,boston children s hospital,218272.0,NaN,NaN,NaN,NaN,left_only
6584,9.545749e+07,university of california berkeley,215495.0,NaN,NaN,NaN,NaN,left_only
970,1.353101e+08,university of wisconsin madison,211906.0,NaN,NaN,NaN,NaN,left_only
307,1.577252e+08,university of illinois at urbana champaign,191121.0,NaN,NaN,NaN,NaN,left_only


### 4. Finish

In [64]:

con.close()

print("Done.")

Done.
